# LeiData WAV 1 — first window

Latest `run_incremental_build_leidata_wav1.sh` run. Stage 0 context is above; the main plot shows **matches on window 1 after the full incremental build** (all templates grown and refined across stages).

In [ ]:
%matplotlib inline

from collections import Counter, defaultdict
from pathlib import Path
import json
import re
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.lines import Line2D

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
RESULT_BASE = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_incremental"
RUN_STAMP_RE = re.compile(r"^\d{8}T\d{6}Z$")


def _resolve_latest_run_dir(base: Path) -> Path:
    base = base.resolve()
    stamped = sorted(
        (p for p in base.iterdir() if p.is_dir() and RUN_STAMP_RE.fullmatch(p.name)),
        key=lambda p: p.name,
    )
    if not stamped:
        raise FileNotFoundError(f"No stamped run under {base}")
    return stamped[-1]


RESULT_DIR = _resolve_latest_run_dir(RESULT_BASE)
STAGE_DIR = RESULT_DIR / "stage_00_w1"
STAGE_MANIFEST_PATH = STAGE_DIR / "manifest.json"
FINAL_MANIFEST_PATH = RESULT_DIR / "manifest.json"
if not STAGE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing stage_00_w1 manifest: {STAGE_MANIFEST_PATH}")
if not FINAL_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing final manifest: {FINAL_MANIFEST_PATH}")

stage = json.loads(STAGE_MANIFEST_PATH.read_text())
stage_log = stage["stage_log"]
stage_templates = {
    item["template_id"]: np.load(STAGE_DIR / item["file"])
    for item in stage["templates"]
}
stage_matchings = stage["matchings"]

final = json.loads(FINAL_MANIFEST_PATH.read_text())
final_templates = {
    item["template_id"]: np.load(RESULT_DIR / item["file"])
    for item in final["templates"]
}
window_id = final["window_ids"][0]
final_matchings_w1 = [m for m in final["matchings"] if m["window_id"] == window_id]

# Back-compat aliases used by stage-0 cells below.
templates = stage_templates
matchings = stage_matchings

print(f"Run:   {RESULT_DIR.name}")
print(f"Stage0: {STAGE_DIR.name}  templates={len(stage_templates)}  matches={len(stage_matchings)}")
print(f"Final:  templates={len(final_templates)}  "
      f"window-1 matches={len(final_matchings_w1)}  "
      f"all-window matches={len(final['matchings'])}")
print(f"Window 1: {window_id}")

## Stage 0 summary

Energy path for the first window: peel (empty dict) → grow new atoms → refine.

In [ ]:
e0 = float(stage_log["energy_before_peel"])
e_grow = float(stage_log["energy_after_grow"])
e_ref = float(stage_log["energy_after_refine"])
matching_counts = Counter(m["template_id"] for m in matchings)

print(f"Energy before:     {e0:.6g}")
print(f"After grow:        {e_grow:.6g}  (removed {e0 - e_grow:.6g}, {100*(e0-e_grow)/e0:.1f}%)")
print(f"After refine:      {e_ref:.6g}  (removed {e0 - e_ref:.6g}, {100*(e0-e_ref)/e0:.1f}%)")
print(f"grow_stop_reason:  {stage_log['grow_stop_reason']}")
print(f"refined:           {stage_log['refined']}")
print("Matches per template:", dict(sorted(matching_counts.items())))

## Learned templates (stage 0)

Color limits are symmetric around zero, scaled to the 99th percentile of absolute magnitude.

In [ ]:
n_templates = len(templates)
n_cols = min(4, max(1, n_templates))
n_rows = int(np.ceil(n_templates / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.5 * n_rows), squeeze=False)
for ax, template_id in zip(axes.flat, sorted(templates)):
    data = templates[template_id]
    clim = np.percentile(np.abs(data), 99) or 1.0
    ax.imshow(data, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-clim, vmax=clim)
    ax.set_title(f"t={template_id}  w={data.shape[1]}  matches={matching_counts[template_id]}")
    ax.set_xlabel("time")
    ax.set_ylabel("freq")
for ax in axes.flat[n_templates:]:
    ax.axis("off")
fig.suptitle(f"Stage 0 templates — {STAGE_DIR.name}", y=1.02)
plt.tight_layout()
plt.show()

## Window 1 — matches from the **final** refined dictionary

Uses all templates after the full incremental build (not stage-0-only). Original whitened spectrogram (faint grayscale) with colored bands for each match on window 1. Solid edge = positive amplitude; dashed = negative.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    row = conn.execute(
        "SELECT npy_path FROM windows WHERE window_id = ?",
        (window_id,),
    ).fetchone()
    if row is None:
        raise KeyError(f"window_id not in catalog: {window_id}")
    npy_path = Path(row["npy_path"])

original = np.asarray(np.load(npy_path), dtype=np.float64)
print(f"Loaded {npy_path.name} shape={original.shape}")

w1_counts = Counter(m["template_id"] for m in final_matchings_w1)
active_ids = sorted(w1_counts)
e_before = float(final["energy_before"])
e_after = float(final["energy_after"])
print(
    f"Final dictionary: {len(final_templates)} templates; "
    f"window-1 matches={len(final_matchings_w1)} "
    f"from {len(active_ids)} atoms; "
    f"stack energy {e_before:.6g} -> {e_after:.6g} "
    f"({100 * (e_before - e_after) / e_before:.1f}% removed)"
)
print("Window-1 matches per template:", dict(w1_counts))

color_cycle = plt.cm.tab20(np.linspace(0, 1, max(20, len(active_ids))))
template_colors = {
    tid: color_cycle[i % len(color_cycle)] for i, tid in enumerate(active_ids)
}

fig, ax = plt.subplots(figsize=(16, 5.5), constrained_layout=True)
vmin, vmax = np.percentile(original, [5, 99.5])
ax.set_facecolor("white")
ax.imshow(
    original,
    aspect="auto",
    origin="lower",
    cmap="Greys",
    vmin=vmin,
    vmax=vmax,
    alpha=0.22,
)

n_freq = original.shape[0]
for item in sorted(final_matchings_w1, key=lambda m: m["t_start"]):
    tid = item["template_id"]
    width = final_templates[tid].shape[1]
    color = template_colors[tid]
    ax.add_patch(
        mpatches.Rectangle(
            (item["t_start"] - 0.5, -0.5),
            width,
            n_freq,
            facecolor=color,
            alpha=0.30,
            edgecolor=color,
            linewidth=1.8,
            linestyle="-" if item["amplitude"] >= 0 else "--",
        )
    )
    ax.annotate(
        str(tid),
        (item["t_start"] + width / 2, n_freq * 0.97),
        color=color,
        ha="center",
        va="top",
        fontsize=9,
        fontweight="bold",
    )

ax.set_title(
    f"Window 1 — final dictionary ({len(final_templates)} templates, "
    f"{len(final_matchings_w1)} matches on this window)\n{window_id}"
)
ax.set_xlabel("Time bin")
ax.set_ylabel("Frequency bin")

legend_handles = [
    mpatches.Patch(
        facecolor=template_colors[tid],
        alpha=0.4,
        edgecolor=template_colors[tid],
        label=(
            f"t={tid} (w={final_templates[tid].shape[1]}, "
            f"{w1_counts[tid]} on w1)"
        ),
    )
    for tid in active_ids
] + [
    Line2D([], [], color="black", linestyle="-", label="positive amplitude"),
    Line2D([], [], color="black", linestyle="--", label="negative amplitude"),
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    ncols=min(5, len(legend_handles)),
    framealpha=0.9,
)
plt.show()